import os
DB_PATH = os.path.expanduser("~/Downloads/HackYourSummer Project#1/nmfp.duckdb")
os.remove(DB_PATH)
print("Deleted")

In [1]:
"""
Step 1: Create the DuckDB database
-----------------------------------
Run this once to set up nmfp.duckdb with the exact schema
from the real SEC N-MFP TSV files.

Join key:         ACCESSION_NUMBER (present in both files)
Fund type filter: MONEYMARKETFUNDCATEGORY = 'Prime'

Usage:
    pip install duckdb
    python create_db.py
"""

import duckdb
import os

DB_PATH = os.path.expanduser("~/Downloads/HackYourSummer Project#1/nmfp.duckdb")


def create_db():
    if os.path.exists(DB_PATH):
        os.remove(DB_PATH)
        print(f"Deleted existing {DB_PATH}\n")

    print(f"Creating {DB_PATH} ...\n")
    con = duckdb.connect(DB_PATH)

    # ── Table 1: series_info ─────────────────────────────────────────────
    # From NMFP_SERIESLEVELINFO.tsv
    # One row per fund series per monthly ZIP.
    # MONEYMARKETFUNDCATEGORY values: 'Prime', 'Government', 'Other Tax Exempt', 'Single State'
    con.execute("""
        CREATE TABLE series_info (
            -- Identity
            ACCESSION_NUMBER                VARCHAR,
            SECURITIESACTFILENUMBER         VARCHAR,

            -- Fund metadata
            INDPPUBACCTNAME                 VARCHAR,   -- independent public accountant
            INDPPUBACCTCITY                 VARCHAR,
            INDPPUBACCTSTATECOUNTRY         VARCHAR,
            FEEDERFUNDFLAG                  VARCHAR,
            MASTERFUNDFLAG                  VARCHAR,
            SERIESFUNDINSUCMPNYSEPACCNTFLA  VARCHAR,
            MONEYMARKETFUNDCATEGORY         VARCHAR,   -- 'Prime' | 'Government' | 'Other Tax Exempt' | 'Single State'
            FUNDEXEMPTRETAILFLAG            VARCHAR,
            FUNDRETAILMONEYMARKETFLAG       VARCHAR,
            GOVMONEYMRKTFUNDFLAG            VARCHAR,

            -- Portfolio maturity
            AVERAGEPORTFOLIOMATURITY        DOUBLE,
            AVERAGELIFEMATURITY             DOUBLE,

            -- Weekly daily liquid assets ($ total, 5 Fridays per month)
            TOTDLYLIQUIDASSETFRIDAYWEEK1    DOUBLE,
            TOTDLYLIQUIDASSETFRIDAYWEEK2    DOUBLE,
            TOTDLYLIQUIDASSETFRIDAYWEEK3    DOUBLE,
            TOTDLYLIQUIDASSETFRIDAYWEEK4    DOUBLE,
            TOTDLYLIQUIDASSETFRIDAYWEEK5    DOUBLE,

            -- Weekly weekly liquid assets ($ total, 5 Fridays)
            TOTWLYLIQUIDASSETFRIDAYWEEK1    DOUBLE,
            TOTWLYLIQUIDASSETFRIDAYWEEK2    DOUBLE,
            TOTWLYLIQUIDASSETFRIDAYWEEK3    DOUBLE,
            TOTWLYLIQUIDASSETFRIDAYWEEK4    DOUBLE,
            TOTWLYLIQUIDASSETFRIDAYWEEK5    DOUBLE,

            -- Daily liquid asset % (5 Fridays)
            PCTDLYLIQUIDASSETFRIDAYWEEK1    DOUBLE,
            PCTDLYLIQUIDASSETFRIDAYWEEK2    DOUBLE,
            PCTDLYLIQUIDASSETFRIDAYWEEK3    DOUBLE,
            PCTDLYLIQUIDASSETFRIDAYWEEK4    DOUBLE,
            PCTDLYLIQUIDASSETFRIDAYWEEK5    DOUBLE,

            -- Weekly liquid asset % (5 Fridays)
            PCTWKLYLIQUIDASSETFRIDAYWEEK1   DOUBLE,
            PCTWKLYLIQUIDASSETFRIDAYWEEK2   DOUBLE,
            PCTWKLYLIQUIDASSETFRIDAYWEEK3   DOUBLE,
            PCTWKLYLIQUIDASSETFRIDAYWEEK4   DOUBLE,
            PCTWKLYLIQUIDASSETFRIDAYWEEK5   DOUBLE,

            -- Portfolio values
            CASH                            DOUBLE,
            TOTALVALUEPORTFOLIOSECURITIES   DOUBLE,
            AMORTIZEDCOSTPORTFOLIOSECURITI  DOUBLE,
            TOTALVALUEOTHERASSETS           DOUBLE,
            TOTALVALUELIABILITIES           DOUBLE,
            NETASSETOFSERIES                DOUBLE,
            NUMBEROFSHARESOUTSTANDING       DOUBLE,

            -- Pricing
            SEEKSTABLEPRICEPERSHARE         VARCHAR,
            STABLEPRICEPERSHARE             DOUBLE,
            SEVENDAYGROSSYIELD              DOUBLE,

            -- NAV per share (5 Fridays)
            NETASSETVALUEFRIDAYWEEK1        DOUBLE,
            NETASSETVALUEFRIDAYWEEK2        DOUBLE,
            NETASSETVALUEFRIDAYWEEK3        DOUBLE,
            NETASSETVALUEFRIDAYWEEK4        DOUBLE,
            NETASSETVALUEFRIDAYWEEK5        DOUBLE,

            -- Flags
            CASHMGMTVEHICLEAFFLIATEDFUNDF   VARCHAR,
            LIQUIDITYFEEFUNDAPPLYFLAG       VARCHAR,

            -- ETL metadata
            SOURCE_PERIOD                   VARCHAR    -- e.g. '2026-04-09 to 2026-05-07'
        )
    """)
    print("Created table: series_info  (51 columns + SOURCE_PERIOD)")

    # ── Table 2: liquidity ───────────────────────────────────────────────
    # From NMFP_LIQUIDASSETSDETAILS.tsv
    # ~20 rows per accession number (one per calendar day in the reporting month)
    con.execute("""
        CREATE TABLE liquidity (
            ACCESSION_NUMBER            VARCHAR,
            TOTVALUEDAILYLIQUIDASSETS   DOUBLE,    -- $ total daily liquid assets
            TOTVALUEWEEKLYLIQUIDASSETS  DOUBLE,    -- $ total weekly liquid assets
            PCTDAILYLIQUIDASSETS        DOUBLE,    -- % of net assets (e.g. 0.9784 = 97.84%)
            PCTWEEKLYLIQUIDASSETS       DOUBLE,    -- % of net assets
            TOTLIQUIDASSETSNEARPCTDATE  VARCHAR,   -- date: 'DD-MON-YYYY' e.g. '15-MAY-2026'

            -- ETL metadata
            SOURCE_PERIOD               VARCHAR
        )
    """)
    print("Created table: liquidity  (6 columns + SOURCE_PERIOD)")

    # ── Table 3: portfolio_securities ────────────────────────────────────
    # From NMFP_SCHPORTFOLIOSECURITIES.tsv
    # One row per holding per fund per period
    con.execute("""
        CREATE TABLE portfolio_securities (
            ACCESSION_NUMBER                    VARCHAR,
            NAMEOFISSUER                        VARCHAR,
            INVESTMENTCATEGORY                  VARCHAR,
            PERCENTAGEOFMONEYMARKETFUNDNET      DOUBLE,
            INCLUDINGVALUEOFANYSPONSORSUPP      DOUBLE,
            DAILYLIQUIDASSETSECURITYFLAG        VARCHAR,
            WEEKLYLIQUIDASSETSECURITYFLAG       VARCHAR,
            INVESTMENTMATURITYDATEWAM           VARCHAR,
            SOURCE_PERIOD                       VARCHAR
        )
    """)
    print("Created table: portfolio_securities  (8 columns + SOURCE_PERIOD)")
    
    # ── Table 4: shareholder_flows ───────────────────────────────────────
    # From NMFP_DLYSHAREHOLDERFLOWREPORT.tsv
    # Aggregated to one row per fund (accession) per period
    # Raw file is class-level daily flows — summed across classes and days
    # Only available in new format ZIPs (June 2024 onward)
    con.execute("""
        CREATE TABLE IF NOT EXISTS shareholder_flows (
            ACCESSION_NUMBER        VARCHAR,
            TOTAL_SUBSCRIPTIONS     DOUBLE,
            TOTAL_REDEMPTIONS       DOUBLE,
            NET_FLOW                DOUBLE,
            SOURCE_PERIOD           VARCHAR
        )
    """)
   # ── Table 5: yield_data ──────────────────────────────────────────────
    # From NMFP_SEVENDAYGROSSYIELD.tsv (new format)
    # or SEVENDAYGROSSYIELD column in series_info (old format)
    # Aggregated to one row per fund per period (average daily yield)
    con.execute("""
        CREATE TABLE yield_data (
            ACCESSION_NUMBER            VARCHAR,
            AVG_SEVEN_DAY_GROSS_YIELD   DOUBLE,
            SOURCE_PERIOD               VARCHAR
        )
    """)


    # ── Indexes ──────────────────────────────────────────────────────────
    con.execute("CREATE INDEX idx_series_accession  ON series_info (ACCESSION_NUMBER)")
    con.execute("CREATE INDEX idx_series_category   ON series_info (MONEYMARKETFUNDCATEGORY)")
    con.execute("CREATE INDEX idx_series_period     ON series_info (SOURCE_PERIOD)")
    con.execute("CREATE INDEX idx_liq_accession     ON liquidity   (ACCESSION_NUMBER)")
    con.execute("CREATE INDEX idx_liq_period        ON liquidity   (SOURCE_PERIOD)")
    con.execute("CREATE INDEX idx_liq_date ON liquidity (TOTLIQUIDASSETSNEARPCTDATE)")
    con.execute("CREATE INDEX idx_port_accession ON portfolio_securities (ACCESSION_NUMBER)")
    con.execute("CREATE INDEX idx_port_period    ON portfolio_securities (SOURCE_PERIOD)")
    con.execute("CREATE INDEX idx_flow_accession ON shareholder_flows (ACCESSION_NUMBER)")
    con.execute("CREATE INDEX idx_flow_period    ON shareholder_flows (SOURCE_PERIOD)")
    con.execute("CREATE INDEX idx_yield_accession ON yield_data (ACCESSION_NUMBER)")
    con.execute("CREATE INDEX idx_yield_period    ON yield_data (SOURCE_PERIOD)")

    print("Created indexes")

    # ── Load tracking table ──────────────────────────────────────────────
    con.execute("""
        CREATE TABLE loaded_periods (
            SOURCE_PERIOD   VARCHAR PRIMARY KEY,
            ZIP_URL         VARCHAR,
            LOADED_AT       TIMESTAMP DEFAULT current_timestamp,
            SERIES_ROWS     INTEGER,
            LIQUIDITY_ROWS  INTEGER
        )
    """)
    print("Created table: loaded_periods\n")

    # ── Verify ───────────────────────────────────────────────────────────
    tables = con.execute("SHOW TABLES").fetchall()
    print("Tables in database:")
    for (t,) in tables:
        cols = con.execute(f"DESCRIBE {t}").fetchall()
        print(f"  {t}  ({len(cols)} columns)")

    con.close()
    size = os.path.getsize(DB_PATH)
    print(f"\nDatabase ready: {os.path.abspath(DB_PATH)}  ({size:,} bytes)")
    print("\nNext step: run etl.py to populate it.")


if __name__ == "__main__":
    create_db()

Creating /Users/saahilkajarekar/Downloads/HackYourSummer Project#1/nmfp.duckdb ...

Created table: series_info  (51 columns + SOURCE_PERIOD)
Created table: liquidity  (6 columns + SOURCE_PERIOD)
Created table: portfolio_securities  (8 columns + SOURCE_PERIOD)
Created indexes
Created table: loaded_periods

Tables in database:
  liquidity  (7 columns)
  loaded_periods  (5 columns)
  portfolio_securities  (9 columns)
  series_info  (52 columns)
  shareholder_flows  (5 columns)
  yield_data  (3 columns)

Database ready: /Users/saahilkajarekar/Downloads/HackYourSummer Project#1/nmfp.duckdb  (536,576 bytes)

Next step: run etl.py to populate it.


In [2]:
import duckdb
import os

DB_PATH = os.path.expanduser("~/Downloads/HackYourSummer Project#1/nmfp.duckdb")

con = duckdb.connect(DB_PATH)

# Show all tables
print("Tables:")
print(con.execute("SHOW TABLES").df())

# Show columns for each table
for table in ["series_info", "liquidity", "loaded_periods"]:
    print(f"\n{table} columns:")
    print(con.execute(f"DESCRIBE {table}").df())

con.close()

Tables:
                   name
0             liquidity
1        loaded_periods
2  portfolio_securities
3           series_info
4     shareholder_flows
5            yield_data

series_info columns:
                       column_name column_type null   key default extra
0                 ACCESSION_NUMBER     VARCHAR  YES  None    None  None
1          SECURITIESACTFILENUMBER     VARCHAR  YES  None    None  None
2                  INDPPUBACCTNAME     VARCHAR  YES  None    None  None
3                  INDPPUBACCTCITY     VARCHAR  YES  None    None  None
4          INDPPUBACCTSTATECOUNTRY     VARCHAR  YES  None    None  None
5                   FEEDERFUNDFLAG     VARCHAR  YES  None    None  None
6                   MASTERFUNDFLAG     VARCHAR  YES  None    None  None
7   SERIESFUNDINSUCMPNYSEPACCNTFLA     VARCHAR  YES  None    None  None
8          MONEYMARKETFUNDCATEGORY     VARCHAR  YES  None    None  None
9             FUNDEXEMPTRETAILFLAG     VARCHAR  YES  None    None  None
10       F

In [4]:
import duckdb, os

DB_PATH = os.path.expanduser("~/Downloads/HackYourSummer Project#1/nmfp.duckdb")
con = duckdb.connect(DB_PATH)

# Row counts
print("series_info rows:", con.execute("SELECT COUNT(*) FROM series_info").fetchone()[0])
print("liquidity rows:  ", con.execute("SELECT COUNT(*) FROM liquidity").fetchone()[0])

# Confirm only Prime funds loaded
print("\nFund categories:")
print(con.execute("SELECT MONEYMARKETFUNDCATEGORY, COUNT(*) as n FROM series_info GROUP BY 1").df())

# Confirm period recorded
print("\nLoaded periods:")
print(con.execute("SELECT * FROM loaded_periods").df())

# Check for NULLs in key columns
print("\nNull counts in liquidity:")
print(con.execute("""
    SELECT 
        COUNT(*) - COUNT(ACCESSION_NUMBER)        AS null_accession,
        COUNT(*) - COUNT(TOTLIQUIDASSETSNEARPCTDATE) AS null_date,
        COUNT(*) - COUNT(PCTDAILYLIQUIDASSETS)    AS null_pct_daily,
        COUNT(*) - COUNT(PCTWEEKLYLIQUIDASSETS)   AS null_pct_weekly
    FROM liquidity
""").df())

# Sample joined data
print("\nSample joined data:")
print(con.execute("""
    SELECT s.ACCESSION_NUMBER, s.INDPPUBACCTNAME, l.TOTLIQUIDASSETSNEARPCTDATE,
           l.PCTDAILYLIQUIDASSETS, l.PCTWEEKLYLIQUIDASSETS
    FROM liquidity l
    JOIN series_info s ON l.ACCESSION_NUMBER = s.ACCESSION_NUMBER
    LIMIT 5
""").df())

con.close()

series_info rows: 0
liquidity rows:   0

Fund categories:
Empty DataFrame
Columns: [MONEYMARKETFUNDCATEGORY, n]
Index: []

Loaded periods:
Empty DataFrame
Columns: [SOURCE_PERIOD, ZIP_URL, LOADED_AT, SERIES_ROWS, LIQUIDITY_ROWS]
Index: []

Null counts in liquidity:
   null_accession  null_date  null_pct_daily  null_pct_weekly
0               0          0               0                0

Sample joined data:
Empty DataFrame
Columns: [ACCESSION_NUMBER, INDPPUBACCTNAME, TOTLIQUIDASSETSNEARPCTDATE, PCTDAILYLIQUIDASSETS, PCTWEEKLYLIQUIDASSETS]
Index: []


In [7]:
"""
Step 3: Full History ETL
------------------------
Downloads all N-MFP ZIPs from June 2022 to present from the SEC DERA page,
filters for ALL fund types (Prime, Government, Tax Exempt, Single State),
and loads into nmfp.duckdb.

Handles both ZIP formats:
  - Old format: NMFP_SERIESLEVELINFO.tsv + NMFP_SCHPORTFOLIOSECURITIES.tsv
  - New format: Above + NMFP_LIQUIDASSETSDETAILS.tsv + NMFP_DLYSHAREHOLDERFLOWREPORT.tsv
                     + NMFP_SEVENDAYGROSSYIELD.tsv

Already-loaded periods are skipped automatically (resume-safe).

Dependencies: pip install requests beautifulsoup4 duckdb pandas
"""

import io
import os
import time
import zipfile

import duckdb
import pandas as pd
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin

# ── Config ───────────────────────────────────────────────────────────────────
DB_PATH  = os.path.expanduser("~/Downloads/HackYourSummer Project#1/nmfp.duckdb")
PAGE_URL = "https://www.sec.gov/data-research/sec-markets-data/dera-form-n-mfp-data-sets"

START_YEAR  = 2022
START_MONTH = 6

DELAY_SEC = 0.6

HEADERS = {
    "User-Agent": "SaahilKajarekar saahil.kajarekar@gmail.com",
    "Accept-Encoding": "identity",
    "Host": "www.sec.gov",
}

SERIES_FILE    = "NMFP_SERIESLEVELINFO.tsv"
LIQUIDITY_FILE = "NMFP_LIQUIDASSETSDETAILS.tsv"
PORTFOLIO_FILE = "NMFP_SCHPORTFOLIOSECURITIES.tsv"
FLOW_FILE      = "NMFP_DLYSHAREHOLDERFLOWREPORT.tsv"
YIELD_FILE     = "NMFP_SEVENDAYGROSSYIELD.tsv"

PORTFOLIO_COLS = [
    "ACCESSION_NUMBER",
    "NAMEOFISSUER",
    "INVESTMENTCATEGORY",
    "PERCENTAGEOFMONEYMARKETFUNDNET",
    "INCLUDINGVALUEOFANYSPONSORSUPP",
    "DAILYLIQUIDASSETSECURITYFLAG",
    "WEEKLYLIQUIDASSETSECURITYFLAG",
    "INVESTMENTMATURITYDATEWAM",
]

# Fund types to include
FUND_TYPES = {"Prime", "Government", "Other Tax Exempt", "Single State"}
# ─────────────────────────────────────────────────────────────────────────────


def get_zip_urls() -> list[dict]:
    """Scrape the SEC page for all ZIP links from June 2022 onward."""
    resp = requests.get(PAGE_URL, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    soup = BeautifulSoup(resp.text, "html.parser")
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if "nmfp.zip" not in href:
            continue
        filename = href.split("/")[-1]
        try:
            year  = int(filename[:4])
            month = int(filename[4:6])
        except ValueError:
            continue
        if (year, month) < (START_YEAR, START_MONTH):
            continue
        label = filename.replace("_nmfp.zip", "")
        links.append({"label": label, "url": urljoin(PAGE_URL, href)})
    links.sort(key=lambda x: x["label"])
    return links


def fetch_zip(url: str) -> zipfile.ZipFile:
    """Download a ZIP into memory and return a ZipFile object."""
    resp = requests.get(url, headers=HEADERS, timeout=120)
    resp.raise_for_status()
    return zipfile.ZipFile(io.BytesIO(resp.content))


def read_tsv_from_zip(zf: zipfile.ZipFile, filename: str) -> pd.DataFrame | None:
    """Read a TSV from an in-memory ZipFile. Returns None if not found."""
    match = next(
        (n for n in zf.namelist() if os.path.basename(n).upper() == filename.upper()),
        None,
    )
    if match is None:
        return None
    with zf.open(match) as f:
        df = pd.read_csv(f, sep="\t", low_memory=False, encoding="utf-8", on_bad_lines="warn")
    df.columns = df.columns.str.strip().str.upper()
    return df


def align_to_schema(con: duckdb.DuckDBPyConnection, table: str, df: pd.DataFrame) -> pd.DataFrame:
    """Add missing columns as NULL, drop extras, reorder to match DB schema."""
    schema_cols = [row[0].upper() for row in con.execute(f"DESCRIBE {table}").fetchall()]
    for col in schema_cols:
        if col not in df.columns:
            df[col] = None
    return df[[c for c in schema_cols if c in df.columns]]


def detect_format(zf: zipfile.ZipFile) -> str:
    """Detect old vs new ZIP format based on presence of liquidity file."""
    names = [os.path.basename(n).upper() for n in zf.namelist()]
    if LIQUIDITY_FILE.upper() in names:
        return "new"
    return "old"


def get_already_loaded(con: duckdb.DuckDBPyConnection) -> set:
    return {row[0] for row in con.execute("SELECT SOURCE_PERIOD FROM loaded_periods").fetchall()}


def load_portfolio(con: duckdb.DuckDBPyConnection, zf: zipfile.ZipFile,
                   accessions: set, label: str) -> int:
    """Load portfolio securities, keeping only selected columns."""
    port_df = read_tsv_from_zip(zf, PORTFOLIO_FILE)
    if port_df is None:
        return 0
    port_df = port_df[port_df["ACCESSION_NUMBER"].isin(accessions)].copy()
    port_df = port_df[[c for c in PORTFOLIO_COLS if c in port_df.columns]]
    port_df["SOURCE_PERIOD"] = label
    port_df = align_to_schema(con, "portfolio_securities", port_df)
    con.execute("INSERT INTO portfolio_securities SELECT * FROM port_df")
    return len(port_df)


def load_shareholder_flows(con: duckdb.DuckDBPyConnection, zf: zipfile.ZipFile,
                           accessions: set, label: str) -> int:
    """Aggregate class-level daily flows to one row per fund per period."""
    flow_df = read_tsv_from_zip(zf, FLOW_FILE)
    if flow_df is None:
        return 0
    flow_df = flow_df[flow_df["ACCESSION_NUMBER"].isin(accessions)].copy()
    if flow_df.empty:
        return 0
    agg = flow_df.groupby("ACCESSION_NUMBER").agg(
        TOTAL_SUBSCRIPTIONS=("DAILYGROSSSUBSCRIPTIONS", "sum"),
        TOTAL_REDEMPTIONS=("DAILYGROSSREDEMPTIONS",     "sum"),
    ).reset_index()
    agg["NET_FLOW"]      = agg["TOTAL_SUBSCRIPTIONS"] - agg["TOTAL_REDEMPTIONS"]
    agg["SOURCE_PERIOD"] = label
    agg = align_to_schema(con, "shareholder_flows", agg)
    con.execute("INSERT INTO shareholder_flows SELECT * FROM agg")
    return len(agg)


def load_yield_new(con: duckdb.DuckDBPyConnection, zf: zipfile.ZipFile,
                   accessions: set, label: str) -> int:
    """Load yield from new format ZIP — aggregate daily to monthly average."""
    yield_df = read_tsv_from_zip(zf, YIELD_FILE)
    if yield_df is None:
        return 0
    yield_df = yield_df[yield_df["ACCESSION_NUMBER"].isin(accessions)].copy()
    if yield_df.empty:
        return 0
    agg = yield_df.groupby("ACCESSION_NUMBER").agg(
        AVG_SEVEN_DAY_GROSS_YIELD=("SEVENDAYGROSSYIELDVALUE", "mean")
    ).reset_index()
    agg["SOURCE_PERIOD"] = label
    agg = align_to_schema(con, "yield_data", agg)
    con.execute("INSERT INTO yield_data SELECT * FROM agg")
    return len(agg)


def load_yield_old(con: duckdb.DuckDBPyConnection, fund_df: pd.DataFrame, label: str) -> int:
    """Load yield from old format — copy from series_info SEVENDAYGROSSYIELD column."""
    old_yield = fund_df[["ACCESSION_NUMBER", "SEVENDAYGROSSYIELD"]].copy()
    old_yield = old_yield[old_yield["SEVENDAYGROSSYIELD"].notna()]
    if old_yield.empty:
        return 0
    old_yield = old_yield.rename(columns={"SEVENDAYGROSSYIELD": "AVG_SEVEN_DAY_GROSS_YIELD"})
    old_yield["SOURCE_PERIOD"] = label
    old_yield = align_to_schema(con, "yield_data", old_yield)
    con.execute("INSERT INTO yield_data SELECT * FROM old_yield")
    return len(old_yield)


def process_zip(con: duckdb.DuckDBPyConnection, zf: zipfile.ZipFile, label: str) -> tuple[int, int]:
    """Extract all fund data from a ZipFile and insert into DB."""
    fmt = detect_format(zf)
    print(f"    Format: {fmt}")

    # ── Series info (all fund types) ──────────────────────────────────────
    series_df = read_tsv_from_zip(zf, SERIES_FILE)
    if series_df is None:
        print(f"    [skip] {SERIES_FILE} not found")
        return 0, 0

    series_df["MONEYMARKETFUNDCATEGORY"] = (
        series_df["MONEYMARKETFUNDCATEGORY"].astype(str).str.strip()
    )

    # Filter to all relevant fund types
    fund_df = series_df[series_df["MONEYMARKETFUNDCATEGORY"].isin(FUND_TYPES)].copy()

    if fund_df.empty:
        print(f"    [skip] No matching funds found")
        return 0, 0

    accessions = set(fund_df["ACCESSION_NUMBER"].dropna().unique())
    fund_df["SOURCE_PERIOD"] = label

    # Print breakdown by fund type
    type_counts = fund_df["MONEYMARKETFUNDCATEGORY"].value_counts().to_dict()
    print(f"    Fund types: {type_counts}")

    fund_df = align_to_schema(con, "series_info", fund_df)
    con.execute("INSERT INTO series_info SELECT * FROM fund_df")
    print(f"    series_info rows : {len(fund_df)}")

    # ── Liquidity (new format only) ───────────────────────────────────────
    liq_rows = 0
    if fmt == "new":
        liq_df = read_tsv_from_zip(zf, LIQUIDITY_FILE)
        if liq_df is not None:
            liq_filtered = liq_df[liq_df["ACCESSION_NUMBER"].isin(accessions)].copy()
            liq_filtered["SOURCE_PERIOD"] = label
            liq_filtered = align_to_schema(con, "liquidity", liq_filtered)
            con.execute("INSERT INTO liquidity SELECT * FROM liq_filtered")
            liq_rows = len(liq_filtered)
            print(f"    liquidity rows   : {liq_rows}")
    else:
        print(f"    liquidity rows   : 0 (old format — Friday cols in series_info)")

    # ── Portfolio securities (both formats) ──────────────────────────────
    port_rows = load_portfolio(con, zf, accessions, label)
    print(f"    portfolio rows   : {port_rows}")

    # ── Shareholder flows (new format only) ──────────────────────────────
    flow_rows = 0
    if fmt == "new":
        flow_rows = load_shareholder_flows(con, zf, accessions, label)
        print(f"    flow rows        : {flow_rows}")
    else:
        print(f"    flow rows        : 0 (old format — not available)")

    # ── Yield data ────────────────────────────────────────────────────────
    if fmt == "new":
        yield_rows = load_yield_new(con, zf, accessions, label)
        print(f"    yield rows       : {yield_rows} (new format)")
    else:
        yield_rows = load_yield_old(con, fund_df, label)
        print(f"    yield rows       : {yield_rows} (old format)")

    # ── Record in loaded_periods ──────────────────────────────────────────
    con.execute(
        "INSERT INTO loaded_periods (SOURCE_PERIOD, ZIP_URL, SERIES_ROWS, LIQUIDITY_ROWS) VALUES (?, ?, ?, ?)",
        [label, "", len(fund_df), liq_rows],
    )

    return len(fund_df), liq_rows


def main():
    print(f"Database: {DB_PATH}")
    print(f"Fetching ZIP list from SEC (from {START_YEAR}-{START_MONTH:02d} onward)...\n")

    zip_links = get_zip_urls()
    if not zip_links:
        print("No ZIPs found.")
        return

    print(f"Found {len(zip_links)} ZIP(s) to process.\n")

    con = duckdb.connect(DB_PATH)
    already_loaded = get_already_loaded(con)
    con.close()

    if already_loaded:
        print(f"Skipping {len(already_loaded)} already-loaded period(s).\n")

    total_series = 0
    total_liq    = 0

    for i, item in enumerate(zip_links, 1):
        label = item["label"]
        print(f"[{i}/{len(zip_links)}] {label}")

        if label in already_loaded:
            print("    [skip] already in DB")
            continue

        try:
            zf  = fetch_zip(item["url"])
            con = duckdb.connect(DB_PATH)
            s_rows, l_rows = process_zip(con, zf, label)
            con.close()

            total_series += s_rows
            total_liq    += l_rows

        except requests.HTTPError as e:
            print(f"    [err] HTTP {e.response.status_code}")
        except Exception as e:
            print(f"    [err] {e}")

        time.sleep(DELAY_SEC)

    print(f"\n{'─'*50}")
    print(f"Total series_info rows inserted : {total_series:,}")
    print(f"Total liquidity rows inserted   : {total_liq:,}")
    print("Done.")


main()

Database: /Users/saahilkajarekar/Downloads/HackYourSummer Project#1/nmfp.duckdb
Fetching ZIP list from SEC (from 2022-06 onward)...

Found 49 ZIP(s) to process.

Skipping 25 already-loaded period(s).

[1/49] 20220701-20220710
    Format: old
    Fund types: {'Prime': 77, 'Other Tax Exempt': 36, 'Single State': 27}
    series_info rows : 140
    liquidity rows   : 0 (old format — Friday cols in series_info)
    portfolio rows   : 19665
    flow rows        : 0 (old format — not available)
    yield rows       : 140 (old format)
[2/49] 20220711-20220805
    Format: old
    Fund types: {'Prime': 77, 'Other Tax Exempt': 36, 'Single State': 27}
    series_info rows : 140
    liquidity rows   : 0 (old format — Friday cols in series_info)
    portfolio rows   : 18983
    flow rows        : 0 (old format — not available)
    yield rows       : 140 (old format)
[3/49] 20220808-20220908
    Format: old
    Fund types: {'Prime': 78, 'Other Tax Exempt': 38, 'Single State': 28}
    series_info rows

In [ ]:
def read_tsv_from_zip(zf: zipfile.ZipFile, filename: str) -> pd.DataFrame | None:
    match = next(
        (n for n in zf.namelist() if os.path.basename(n).upper() == filename.upper()),
        None,
    )
    if match is None:
        # Print what IS in the ZIP so we can see the actual filename
        print(f"    [files in ZIP]: {zf.namelist()}")
        return None
    with zf.open(match) as f:
        df = pd.read_csv(f, sep="\t", low_memory=False, encoding="utf-8", on_bad_lines="warn")
    df.columns = df.columns.str.strip().str.upper()
    return df

In [ ]:
import duckdb
import os
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import StandardScaler

DB_PATH = os.path.expanduser("~/Downloads/HackYourSummer Project#1/nmfp.duckdb")
con = duckdb.connect(DB_PATH)

# ── Step 1: Series info with stable fund ID ──────────────────────────────────
series = con.execute("""
    SELECT
        SECURITIESACTFILENUMBER         as FUND_ID,
        ACCESSION_NUMBER,
        SOURCE_PERIOD,
        AVERAGEPORTFOLIOMATURITY        as WAM,
        AVERAGELIFEMATURITY             as WAL,
        PCTDLYLIQUIDASSETFRIDAYWEEK1    as DAILY_LIQ_PCT_OLD,
        PCTWKLYLIQUIDASSETFRIDAYWEEK1   as WEEKLY_LIQ_PCT_OLD,
        NETASSETOFSERIES                as NET_ASSETS,
        NUMBEROFSHARESOUTSTANDING       as SHARES
    FROM series_info
    WHERE SECURITIESACTFILENUMBER IS NOT NULL
""").df()

# ── Step 2: Liquidity from new format (liquidity table) ──────────────────────
liq = con.execute("""
    SELECT 
        ACCESSION_NUMBER,
        SOURCE_PERIOD,
        AVG(PCTDAILYLIQUIDASSETS)  as DAILY_LIQ_PCT_NEW,
        AVG(PCTWEEKLYLIQUIDASSETS) as WEEKLY_LIQ_PCT_NEW
    FROM liquidity
    GROUP BY ACCESSION_NUMBER, SOURCE_PERIOD
""").df()

# ── Step 3: Issuer concentration ─────────────────────────────────────────────
concentration = con.execute("""
    SELECT 
        ACCESSION_NUMBER,
        SOURCE_PERIOD,
        MAX(PERCENTAGEOFMONEYMARKETFUNDNET) as MAX_ISSUER_PCT
    FROM portfolio_securities
    GROUP BY ACCESSION_NUMBER, SOURCE_PERIOD
""").df()

# ── Step 4: Shareholder flows ─────────────────────────────────────────────────
flows = con.execute("""
    SELECT ACCESSION_NUMBER, SOURCE_PERIOD, NET_FLOW
    FROM shareholder_flows
""").df()

con.close()

# ── Step 5: Merge everything ──────────────────────────────────────────────────
df = series.merge(liq,           on=["ACCESSION_NUMBER", "SOURCE_PERIOD"], how="left")
df = df.merge(concentration,     on=["ACCESSION_NUMBER", "SOURCE_PERIOD"], how="left")
df = df.merge(flows,             on=["ACCESSION_NUMBER", "SOURCE_PERIOD"], how="left")

# Combine old and new liquidity columns
df["DAILY_LIQ_PCT"]  = df["DAILY_LIQ_PCT_NEW"].fillna(df["DAILY_LIQ_PCT_OLD"])
df["WEEKLY_LIQ_PCT"] = df["WEEKLY_LIQ_PCT_NEW"].fillna(df["WEEKLY_LIQ_PCT_OLD"])

# ── Step 6: Outcome variables ─────────────────────────────────────────────────
# Sort by fund and period for time-series calculations
df = df.sort_values(["FUND_ID", "SOURCE_PERIOD"]).reset_index(drop=True)

# Shares change % (proxy for all 48 periods)
df["SHARES_CHANGE_PCT"] = df.groupby("FUND_ID")["SHARES"].pct_change()

# Net flow % of net assets (new periods only)
df["NET_FLOW_PCT"] = df["NET_FLOW"] / df["NET_ASSETS"]

print(f"Total rows: {len(df)}")
print(f"Rows with DAILY_LIQ_PCT:   {df['DAILY_LIQ_PCT'].notna().sum()}")
print(f"Rows with WEEKLY_LIQ_PCT:  {df['WEEKLY_LIQ_PCT'].notna().sum()}")
print(f"Rows with shares change:   {df['SHARES_CHANGE_PCT'].notna().sum()}")
print(f"Rows with flow data:       {df['NET_FLOW_PCT'].notna().sum()}")
print(f"\nSample:")
print(df[["FUND_ID", "SOURCE_PERIOD", "WAM", "WAL",
          "DAILY_LIQ_PCT", "WEEKLY_LIQ_PCT", "MAX_ISSUER_PCT",
          "SHARES_CHANGE_PCT", "NET_FLOW_PCT"]].head(10))

In [ ]:
con = duckdb.connect(DB_PATH)

print("=== Combined risk factors (most recent period) ===")
print(con.execute("""
    WITH liq AS (
        SELECT 
            ACCESSION_NUMBER,
            AVG(PCTDAILYLIQUIDASSETS)  as DAILY_LIQ_PCT,
            AVG(PCTWEEKLYLIQUIDASSETS) as WEEKLY_LIQ_PCT
        FROM liquidity
        WHERE SOURCE_PERIOD = (SELECT MAX(SOURCE_PERIOD) FROM liquidity)
        GROUP BY ACCESSION_NUMBER
    ),
    conc AS (
        SELECT
            ACCESSION_NUMBER,
            MAX(PERCENTAGEOFMONEYMARKETFUNDNET) as MAX_ISSUER_PCT
        FROM portfolio_securities
        WHERE SOURCE_PERIOD = (SELECT MAX(SOURCE_PERIOD) FROM portfolio_securities)
        GROUP BY ACCESSION_NUMBER
    )
    SELECT
        s.SECURITIESACTFILENUMBER   as FUND_ID,
        s.AVERAGEPORTFOLIOMATURITY  as WAM,
        s.AVERAGELIFEMATURITY       as WAL,
        l.DAILY_LIQ_PCT,
        l.WEEKLY_LIQ_PCT,
        c.MAX_ISSUER_PCT,
        s.SEVENDAYGROSSYIELD        as YIELD
    FROM series_info s
    LEFT JOIN liq  l ON s.ACCESSION_NUMBER = l.ACCESSION_NUMBER
    LEFT JOIN conc c ON s.ACCESSION_NUMBER = c.ACCESSION_NUMBER
    WHERE s.SOURCE_PERIOD = (SELECT MAX(SOURCE_PERIOD) FROM series_info)
    AND s.SECURITIESACTFILENUMBER IS NOT NULL
    ORDER BY s.AVERAGEPORTFOLIOMATURITY DESC
    LIMIT 15
""").df())

# Variance across all factors using combined sources
print("\n=== Std dev per factor (new periods, combined sources) ===")
print(con.execute("""
    WITH liq AS (
        SELECT 
            ACCESSION_NUMBER,
            SOURCE_PERIOD,
            AVG(PCTDAILYLIQUIDASSETS)  as DAILY_LIQ_PCT,
            AVG(PCTWEEKLYLIQUIDASSETS) as WEEKLY_LIQ_PCT
        FROM liquidity
        GROUP BY ACCESSION_NUMBER, SOURCE_PERIOD
    ),
    conc AS (
        SELECT
            ACCESSION_NUMBER,
            SOURCE_PERIOD,
            MAX(PERCENTAGEOFMONEYMARKETFUNDNET) as MAX_ISSUER_PCT
        FROM portfolio_securities
        GROUP BY ACCESSION_NUMBER, SOURCE_PERIOD
    )
    SELECT
        ROUND(STDDEV(s.AVERAGEPORTFOLIOMATURITY), 2)  as WAM_std,
        ROUND(STDDEV(s.AVERAGELIFEMATURITY), 2)       as WAL_std,
        ROUND(STDDEV(l.DAILY_LIQ_PCT), 4)             as DAILY_LIQ_std,
        ROUND(STDDEV(l.WEEKLY_LIQ_PCT), 4)            as WEEKLY_LIQ_std,
        ROUND(STDDEV(c.MAX_ISSUER_PCT), 4)            as ISSUER_CONC_std
    FROM series_info s
    LEFT JOIN liq  l ON s.ACCESSION_NUMBER = l.ACCESSION_NUMBER 
                     AND s.SOURCE_PERIOD = l.SOURCE_PERIOD
    LEFT JOIN conc c ON s.ACCESSION_NUMBER = c.ACCESSION_NUMBER
                     AND s.SOURCE_PERIOD = c.SOURCE_PERIOD
    WHERE s.SOURCE_PERIOD >= '20240610'
""").df())

con.close()

In [8]:
import duckdb
import os
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

DB_PATH = os.path.expanduser("~/Downloads/HackYourSummer Project#1/nmfp.duckdb")
con = duckdb.connect(DB_PATH)

# ── Pull all risk factors ─────────────────────────────────────────────────────
series = con.execute("""
    SELECT
        s.SECURITIESACTFILENUMBER       as FUND_ID,
        s.ACCESSION_NUMBER,
        s.SOURCE_PERIOD,
        s.MONEYMARKETFUNDCATEGORY       as FUND_TYPE,
        s.AVERAGEPORTFOLIOMATURITY      as WAM,
        s.AVERAGELIFEMATURITY           as WAL,
        s.PCTDLYLIQUIDASSETFRIDAYWEEK1  as DAILY_LIQ_OLD,
        s.PCTWKLYLIQUIDASSETFRIDAYWEEK1 as WEEKLY_LIQ_OLD
    FROM series_info s
    WHERE s.SECURITIESACTFILENUMBER IS NOT NULL
""").df()

liq = con.execute("""
    SELECT
        ACCESSION_NUMBER, SOURCE_PERIOD,
        AVG(PCTDAILYLIQUIDASSETS)  as DAILY_LIQ_NEW,
        AVG(PCTWEEKLYLIQUIDASSETS) as WEEKLY_LIQ_NEW
    FROM liquidity
    GROUP BY ACCESSION_NUMBER, SOURCE_PERIOD
""").df()

conc = con.execute("""
    SELECT
        ACCESSION_NUMBER, SOURCE_PERIOD,
        MAX(PERCENTAGEOFMONEYMARKETFUNDNET) as MAX_ISSUER_PCT
    FROM portfolio_securities
    GROUP BY ACCESSION_NUMBER, SOURCE_PERIOD
""").df()

con.close()

# ── Merge and combine liquidity sources ───────────────────────────────────────
df = series.merge(liq,  on=["ACCESSION_NUMBER", "SOURCE_PERIOD"], how="left")
df = df.merge(conc, on=["ACCESSION_NUMBER", "SOURCE_PERIOD"], how="left")

df["DAILY_LIQ"]  = df["DAILY_LIQ_NEW"].fillna(df["DAILY_LIQ_OLD"])
df["WEEKLY_LIQ"] = df["WEEKLY_LIQ_NEW"].fillna(df["WEEKLY_LIQ_OLD"])

# Remove problematic fund
df = df[df["FUND_ID"] != "033-49552"].copy()

# Flip liquidity so higher = riskier
df["DAILY_LIQ_RISK"]  = 1 - df["DAILY_LIQ"]
df["WEEKLY_LIQ_RISK"] = 1 - df["WEEKLY_LIQ"]

FEATURES = ["WAM", "WAL", "DAILY_LIQ_RISK", "WEEKLY_LIQ_RISK", "MAX_ISSUER_PCT"]

# ── Train/test split ──────────────────────────────────────────────────────────
# Temporal split: train on older periods, test on recent
TRAIN_CUTOFF = "20250101"   # train: July 2022 → Dec 2024, test: Jan 2025 → present

train_df = df[df["SOURCE_PERIOD"] <  TRAIN_CUTOFF].copy()
test_df  = df[df["SOURCE_PERIOD"] >= TRAIN_CUTOFF].copy()

print(f"Train periods: {train_df['SOURCE_PERIOD'].nunique()}")
print(f"Test periods:  {test_df['SOURCE_PERIOD'].nunique()}")
print(f"Train rows:    {len(train_df):,}")
print(f"Test rows:     {len(test_df):,}")

# ── PCA per fund type ─────────────────────────────────────────────────────────
fund_types = df["FUND_TYPE"].unique()
pca_weights = {}   # store weights per fund type

print("\n" + "="*60)
for fund_type in sorted(fund_types):
    print(f"\n=== {fund_type} ===")

    train_type = train_df[train_df["FUND_TYPE"] == fund_type][FEATURES].dropna()
    test_type  = test_df[test_df["FUND_TYPE"]   == fund_type][FEATURES].dropna()

    if len(train_type) < 25:
        print(f"  [skip] Not enough training data ({len(train_type)} rows)")
        continue

    print(f"  Train rows: {len(train_type):,}  |  Test rows: {len(test_type):,}")

    # Standardize using train statistics only
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_type)
    X_test  = scaler.transform(test_type) if len(test_type) > 0 else None

    # Fit PCA on training data
    pca = PCA(n_components=5)
    pca.fit(X_train)

    # Explained variance
    print(f"  Explained variance by component:")
    for i, var in enumerate(pca.explained_variance_ratio_):
        print(f"    PC{i+1}: {var*100:.1f}%")

    # Derive weights from PC1 loadings
    abs_loadings = np.abs(pca.components_[0])
    weights = abs_loadings / abs_loadings.sum()

    print(f"  PC1 derived weights:")
    for factor, weight in zip(FEATURES, weights):
        print(f"    {factor:<20}: {weight*100:.1f}%")

    pca_weights[fund_type] = {
        "weights":  dict(zip(FEATURES, weights)),
        "scaler":   scaler,
        "pca":      pca,
    }

    # ── Test set validation ───────────────────────────────────────────────
    if X_test is not None and len(X_test) > 0:
        # Project test data onto PC1
        train_scores = pca.transform(X_train)[:, 0]
        test_scores  = pca.transform(X_test)[:, 0]

        # Check that test distribution is similar to train
        print(f"  PC1 score — Train mean: {train_scores.mean():.3f}, std: {train_scores.std():.3f}")
        print(f"  PC1 score — Test  mean: {test_scores.mean():.3f}, std: {test_scores.std():.3f}")

        # KS test: checks if train and test come from same distribution
        from scipy import stats
        ks_stat, ks_pval = stats.ks_2samp(train_scores, test_scores)
        print(f"  KS test: stat={ks_stat:.3f}, p={ks_pval:.3f} ", end="")
        print("✅ distributions match" if ks_pval > 0.05 else "⚠️ distributions differ")

print("\n" + "="*60)
print("\nFinal PCA weights by fund type:")
for fund_type, data in pca_weights.items():
    print(f"\n{fund_type}:")
    for factor, weight in data["weights"].items():
        print(f"  {factor:<20}: {weight*100:.1f}%")

Train periods: 31
Test periods:  18
Train rows:    4,856
Test rows:     5,168


=== Government ===
  Train rows: 1,570  |  Test rows: 3,846
  Explained variance by component:
    PC1: 46.1%
    PC2: 36.7%
    PC3: 8.2%
    PC4: 5.0%
    PC5: 3.9%
  PC1 derived weights:
    WAM                 : 29.4%
    WAL                 : 31.3%
    DAILY_LIQ_RISK      : 4.2%
    WEEKLY_LIQ_RISK     : 5.9%
    MAX_ISSUER_PCT      : 29.2%
  PC1 score — Train mean: -0.000, std: 1.518
  PC1 score — Test  mean: 0.174, std: 1.729
  KS test: stat=0.191, p=0.000 ⚠️ distributions differ

=== Other Tax Exempt ===
  Train rows: 444  |  Test rows: 242
  Explained variance by component:
    PC1: 52.9%
    PC2: 24.3%
    PC3: 16.1%
    PC4: 6.6%
    PC5: 0.0%
  PC1 derived weights:
    WAM                 : 30.2%
    WAL                 : 30.2%
    DAILY_LIQ_RISK      : 7.7%
    WEEKLY_LIQ_RISK     : 25.1%
    MAX_ISSUER_PCT      : 6.9%
  PC1 score — Train mean: 0.000, std: 1.627
  PC1 score — Test  mean: 0.873,

In [23]:
import duckdb
import os
import pandas as pd
import numpy as np

DB_PATH = os.path.expanduser("~/Downloads/HackYourSummer Project#1/nmfp.duckdb")
con = duckdb.connect(DB_PATH)

# ── Pull all data ─────────────────────────────────────────────────────────────
series = con.execute("""
    SELECT
        s.SECURITIESACTFILENUMBER       as FUND_ID,
        s.ACCESSION_NUMBER,
        s.SOURCE_PERIOD,
        s.MONEYMARKETFUNDCATEGORY       as FUND_TYPE,
        s.AVERAGEPORTFOLIOMATURITY      as WAM,
        s.AVERAGELIFEMATURITY           as WAL,
        s.PCTDLYLIQUIDASSETFRIDAYWEEK1  as DAILY_LIQ_OLD,
        s.PCTWKLYLIQUIDASSETFRIDAYWEEK1 as WEEKLY_LIQ_OLD,
        s.NETASSETOFSERIES              as NET_ASSETS,
        s.INDPPUBACCTNAME               as AUDITOR
    FROM series_info s
    WHERE s.SECURITIESACTFILENUMBER IS NOT NULL
""").df()

liq = con.execute("""
    SELECT ACCESSION_NUMBER, SOURCE_PERIOD,
           AVG(PCTDAILYLIQUIDASSETS)  as DAILY_LIQ_NEW,
           AVG(PCTWEEKLYLIQUIDASSETS) as WEEKLY_LIQ_NEW
    FROM liquidity
    GROUP BY ACCESSION_NUMBER, SOURCE_PERIOD
""").df()

# Fix: sum per issuer first, then take max — true issuer concentration
conc = con.execute("""
    SELECT ACCESSION_NUMBER, SOURCE_PERIOD,
           MAX(ISSUER_TOTAL_PCT) as MAX_ISSUER_PCT
    FROM (
        SELECT ACCESSION_NUMBER, SOURCE_PERIOD,
               NAMEOFISSUER,
               SUM(PERCENTAGEOFMONEYMARKETFUNDNET) as ISSUER_TOTAL_PCT
        FROM portfolio_securities
        GROUP BY ACCESSION_NUMBER, SOURCE_PERIOD, NAMEOFISSUER
    )
    GROUP BY ACCESSION_NUMBER, SOURCE_PERIOD
""").df()

yield_df = con.execute("""
    SELECT ACCESSION_NUMBER, SOURCE_PERIOD,
           AVG_SEVEN_DAY_GROSS_YIELD as YIELD
    FROM yield_data
""").df()

con.close()

# ── Merge ─────────────────────────────────────────────────────────────────────
df = series.merge(liq,      on=["ACCESSION_NUMBER", "SOURCE_PERIOD"], how="left")
df = df.merge(conc,         on=["ACCESSION_NUMBER", "SOURCE_PERIOD"], how="left")
df = df.merge(yield_df,     on=["ACCESSION_NUMBER", "SOURCE_PERIOD"], how="left")

df["DAILY_LIQ"]  = df["DAILY_LIQ_NEW"].fillna(df["DAILY_LIQ_OLD"])
df["WEEKLY_LIQ"] = df["WEEKLY_LIQ_NEW"].fillna(df["WEEKLY_LIQ_OLD"])
df = df[df["FUND_ID"] != "033-49552"].copy()
df = df.reset_index(drop=True)

# ── PCA-derived weights per fund type ────────────────────────────────────────
PCA_WEIGHTS = {
    "Government": {
        "WAM":            0.294,
        "WAL":            0.313,
        "DAILY_LIQ":      0.042,
        "WEEKLY_LIQ":     0.059,
        "MAX_ISSUER_PCT": 0.292,
    },
    "Other Tax Exempt": {
        "WAM":            0.302,
        "WAL":            0.302,
        "DAILY_LIQ":      0.000,   # exempt — zero weight
        "WEEKLY_LIQ":     0.251,
        "MAX_ISSUER_PCT": 0.069,
    },
    "Prime": {
        "WAM":            0.225,
        "WAL":            0.234,
        "DAILY_LIQ":      0.183,
        "WEEKLY_LIQ":     0.167,
        "MAX_ISSUER_PCT": 0.191,
    },
    "Single State": {
        "WAM":            0.249,
        "WAL":            0.249,
        "DAILY_LIQ":      0.000,   # exempt — zero weight
        "WEEKLY_LIQ":     0.225,
        "MAX_ISSUER_PCT": 0.088,
    },
}

# ── Regulatory limits per fund type (Rule 2a-7, 2023 reforms) ────────────────
# None = factor not applicable / no regulatory requirement for this fund type
LIMITS = {
    "Prime": {
        "WAM":            {"ideal": 0,   "worst": 60,   "direction": "higher_worse"},
        "WAL":            {"ideal": 0,   "worst": 120,  "direction": "higher_worse"},
        "DAILY_LIQ":      {"ideal": 1.0, "worst": 0.25, "direction": "lower_worse"},
        "WEEKLY_LIQ":     {"ideal": 1.0, "worst": 0.50, "direction": "lower_worse"},
        "MAX_ISSUER_PCT": {"ideal": 0,   "worst": 0.05, "direction": "higher_worse"},
    },
    "Government": {
        "WAM":            {"ideal": 0,   "worst": 60,   "direction": "higher_worse"},
        "WAL":            {"ideal": 0,   "worst": 120,  "direction": "higher_worse"},
        "DAILY_LIQ":      {"ideal": 1.0, "worst": 0.25, "direction": "lower_worse"},
        "WEEKLY_LIQ":     {"ideal": 1.0, "worst": 0.50, "direction": "lower_worse"},
        "MAX_ISSUER_PCT": {"ideal": 0,   "worst": 1.5,  "direction": "higher_worse"},  # no limit
    },
    "Other Tax Exempt": {
        "WAM":            {"ideal": 0,   "worst": 60,   "direction": "higher_worse"},
        "WAL":            {"ideal": 0,   "worst": 120,  "direction": "higher_worse"},
        "DAILY_LIQ":      None,                                                          # exempt
        "WEEKLY_LIQ":     {"ideal": 1.0, "worst": 0.50, "direction": "lower_worse"},
        "MAX_ISSUER_PCT": {"ideal": 0,   "worst": 0.05, "direction": "higher_worse"},
    },
    "Single State": {
        "WAM":            {"ideal": 0,   "worst": 60,   "direction": "higher_worse"},
        "WAL":            {"ideal": 0,   "worst": 120,  "direction": "higher_worse"},
        "DAILY_LIQ":      None,                                                          # exempt
        "WEEKLY_LIQ":     {"ideal": 1.0, "worst": 0.50, "direction": "lower_worse"},
        "MAX_ISSUER_PCT": {"ideal": 0,   "worst": 1.5,  "direction": "higher_worse"},  # no limit
    },
}

# ── Initialize score and violation columns ────────────────────────────────────
score_cols = ["WAM_SCORE", "WAL_SCORE", "DAILY_LIQ_SCORE",
              "WEEKLY_LIQ_SCORE", "MAX_ISSUER_PCT_SCORE"]
viol_cols  = ["WAM_VIOLATION", "WAL_VIOLATION", "DAILY_LIQ_VIOLATION",
              "WEEKLY_LIQ_VIOLATION", "MAX_ISSUER_PCT_VIOLATION"]

for col in score_cols:
    df[col] = np.nan
for col in viol_cols:
    df[col] = False

# ── Score each factor per row using fund-type-specific limits ─────────────────
print("Scoring factors...")
for idx, row in df.iterrows():
    fund_type   = row["FUND_TYPE"]
    fund_limits = LIMITS.get(fund_type, LIMITS["Prime"])

    for factor, params in fund_limits.items():
        # Skip factors not applicable for this fund type
        if params is None:
            continue

        val = row[factor]
        if pd.isna(val):
            continue

        ideal = params["ideal"]
        worst = params["worst"]

        if params["direction"] == "higher_worse":
            raw  = (val - ideal) / (worst - ideal) * 100
            viol = (val > worst) and (worst < 1.5)  # only flag if there's a real regulatory limit
        else:
            raw  = (ideal - val) / (ideal - worst) * 100
            viol = val < worst

        df.at[idx, f"{factor}_SCORE"]     = float(np.clip(raw, 0, 100))
        df.at[idx, f"{factor}_VIOLATION"] = bool(viol)

print("Done scoring.")

# ── Composite risk score using PCA weights per fund type ──────────────────────
FACTOR_TO_SCORE = {
    "WAM":            "WAM_SCORE",
    "WAL":            "WAL_SCORE",
    "DAILY_LIQ":      "DAILY_LIQ_SCORE",
    "WEEKLY_LIQ":     "WEEKLY_LIQ_SCORE",
    "MAX_ISSUER_PCT": "MAX_ISSUER_PCT_SCORE",
}

def compute_risk_score(row):
    fund_type = row["FUND_TYPE"]
    weights   = PCA_WEIGHTS.get(fund_type, PCA_WEIGHTS["Prime"])

    score        = 0
    total_weight = 0

    for factor, weight in weights.items():
        if weight == 0:
            continue   # skip exempt factors
        score_col = FACTOR_TO_SCORE[factor]
        val = row.get(score_col, np.nan)
        if pd.notna(val):
            score        += val * weight
            total_weight += weight

    # Reweight proportionally if any factors are missing
    if total_weight > 0 and total_weight < 1:
        score = score / total_weight

    return score if total_weight > 0 else np.nan

df["RISK_SCORE"] = df.apply(compute_risk_score, axis=1)

# ── Violation flag ────────────────────────────────────────────────────────────
df["HAS_VIOLATION"] = df[viol_cols].any(axis=1)
df["VIOLATION_DETAILS"] = df.apply(lambda row: ", ".join([
    f.replace("_VIOLATION", "")
    for f in viol_cols
    if pd.notna(row[f]) and row[f]
]), axis=1)
df["VIOLATION_DETAILS"] = df["VIOLATION_DETAILS"].replace("", "No Violations")

# ── Risk tier ─────────────────────────────────────────────────────────────────
def risk_tier(score):
    if pd.isna(score):    return None
    if score >= 75:       return "High"
    elif score >= 50:     return "Medium-High"
    elif score >= 25:     return "Medium-Low"
    else:                 return "Low"

df["RISK_TIER"] = df["RISK_SCORE"].apply(risk_tier)

# ── Peer rank within fund type per period ─────────────────────────────────────
df["PEER_RANK_PCT"] = df.groupby(
    ["FUND_TYPE", "SOURCE_PERIOD"]
)["RISK_SCORE"].rank(pct=True) * 100

# ── Preview most recent period ────────────────────────────────────────────────
most_recent = df[df["SOURCE_PERIOD"] == df["SOURCE_PERIOD"].max()].copy()
most_recent = most_recent.sort_values(["FUND_TYPE", "RISK_SCORE"], ascending=[True, False])

print("\n=== Risk Scores — Most Recent Period ===")
print(most_recent[[
    "FUND_ID", "FUND_TYPE", "WAM", "WAL",
    "DAILY_LIQ", "WEEKLY_LIQ", "MAX_ISSUER_PCT",
    "YIELD", "RISK_SCORE", "RISK_TIER",
    "PEER_RANK_PCT", "HAS_VIOLATION", "VIOLATION_DETAILS"
]].to_string(index=False))

print("\n=== Score Distribution by Fund Type ===")
print(df.groupby("FUND_TYPE")["RISK_TIER"].value_counts().to_string())

print("\n=== Average Risk Score by Fund Type ===")
print(df.groupby("FUND_TYPE")["RISK_SCORE"].mean().round(1).to_string())

print("\n=== Violations ===")
violations = df[df["HAS_VIOLATION"] == True]
if violations.empty:
    print("No regulatory violations found.")
else:
    print(violations[["FUND_ID", "SOURCE_PERIOD", "FUND_TYPE", "VIOLATION_DETAILS"]])

Scoring factors...
Done scoring.

=== Risk Scores — Most Recent Period ===
   FUND_ID        FUND_TYPE  WAM   WAL  DAILY_LIQ  WEEKLY_LIQ  MAX_ISSUER_PCT    YIELD  RISK_SCORE   RISK_TIER  PEER_RANK_PCT  HAS_VIOLATION     VIOLATION_DETAILS
 002-58287       Government 53.0 105.0   0.998390    0.998390          1.0249 0.037152   73.336892 Medium-High     100.000000          False         No Violations
 033-31894       Government 53.0  92.0   1.000000    1.000000          1.0189 0.036995   69.801253 Medium-High      99.543379          False         No Violations
333-233596       Government 53.0 112.0   0.698848    0.777933          0.5295 0.036833   69.797773 Medium-High      99.086758          False         No Violations
333-260527       Government 47.0 102.0   0.987238    0.987238          1.0127 0.037019   69.570950 Medium-High      98.630137          False         No Violations
 333-74295       Government 54.0 102.0   0.998343    0.998343          0.8092 0.037043   68.846261 Medium-High

In [24]:
con = duckdb.connect(DB_PATH)

# Check the top issuers for a Prime fund that has a violation
prime_violations = df[
    (df["FUND_TYPE"] == "Prime") & 
    (df["MAX_ISSUER_PCT_VIOLATION"] == True) &
    (df["SOURCE_PERIOD"] == df["SOURCE_PERIOD"].max())
]["ACCESSION_NUMBER"].tolist()

print(f"Prime funds with MAX_ISSUER_PCT violation in most recent period: {len(prime_violations)}")

if prime_violations:
    # Show top issuers for the first violating fund
    print("\nTop issuers for first violating fund:")
    print(con.execute(f"""
        SELECT NAMEOFISSUER,
               SUM(PERCENTAGEOFMONEYMARKETFUNDNET) as TOTAL_PCT,
               COUNT(*) as num_holdings
        FROM portfolio_securities
        WHERE ACCESSION_NUMBER = '{prime_violations[0]}'
        GROUP BY NAMEOFISSUER
        ORDER BY TOTAL_PCT DESC
        LIMIT 10
    """).df())

    # Show the MAX_ISSUER_PCT for all violating funds
    print("\nAll Prime violations in most recent period:")
    print(df[
        (df["FUND_TYPE"] == "Prime") &
        (df["MAX_ISSUER_PCT_VIOLATION"] == True) &
        (df["SOURCE_PERIOD"] == df["SOURCE_PERIOD"].max())
    ][["FUND_ID", "MAX_ISSUER_PCT", "MAX_ISSUER_PCT_SCORE"]].to_string())

con.close()

Prime funds with MAX_ISSUER_PCT violation in most recent period: 32

Top issuers for first violating fund:
                                        NAMEOFISSUER  TOTAL_PCT  num_holdings
0                                Bank of Nova Scotia     0.1205             4
1                                        BNP Paribas     0.0470             3
2  Canadian Imperial Bank of Commerce, Toronto Br...     0.0463             1
3                            TD Securities (USA) LLC     0.0463             2
4                      MUFG Securities Americas Inc.     0.0463             2
5                         J.P. Morgan Securities LLC     0.0448             4
6  Credit Agricole Corporate and Investment Bank,...     0.0445             1
7                                   Societe Generale     0.0445             1
8                        HSBC Securities (USA), Inc.     0.0412             3
9                       Citigroup Global Markets Inc     0.0325             6

All Prime violations in most recen

In [26]:
# One row per fund per period (average share classes)
df_dedup = df.groupby(["FUND_ID", "SOURCE_PERIOD", "FUND_TYPE"]).agg(
    WAM=("WAM", "mean"),
    WAL=("WAL", "mean"),
    DAILY_LIQ=("DAILY_LIQ", "mean"),
    WEEKLY_LIQ=("WEEKLY_LIQ", "mean"),
    MAX_ISSUER_PCT=("MAX_ISSUER_PCT", "mean"),
    YIELD=("YIELD", "mean"),
    NET_ASSETS=("NET_ASSETS", "sum"),
    RISK_SCORE=("RISK_SCORE", "mean"),
    HAS_VIOLATION=("HAS_VIOLATION", "any"),
    VIOLATION_DETAILS=("VIOLATION_DETAILS", "first"),
    RISK_TIER=("RISK_TIER", "first"),
).reset_index()

# Peer rank within fund type per period
df_dedup["PEER_RANK_PCT"] = df_dedup.groupby(
    ["FUND_TYPE", "SOURCE_PERIOD"]
)["RISK_SCORE"].rank(pct=True) * 100

# Display grouped by fund type
most_recent = df_dedup[
    df_dedup["SOURCE_PERIOD"] == df_dedup["SOURCE_PERIOD"].max()
].sort_values(["FUND_TYPE", "RISK_SCORE"], ascending=[True, False])

for fund_type, group in most_recent.groupby("FUND_TYPE"):
    print(f"\n{'='*60}")
    print(f"FUND TYPE: {fund_type}")
    print(f"{'='*60}")
    print(group[[
        "FUND_ID", "WAM", "WAL", "DAILY_LIQ", "WEEKLY_LIQ",
        "MAX_ISSUER_PCT", "YIELD", "RISK_SCORE", "RISK_TIER",
        "PEER_RANK_PCT", "HAS_VIOLATION"
    ]].to_string(index=False))


FUND TYPE: Government
   FUND_ID       WAM        WAL  DAILY_LIQ  WEEKLY_LIQ  MAX_ISSUER_PCT    YIELD  RISK_SCORE   RISK_TIER  PEER_RANK_PCT  HAS_VIOLATION
333-233596 53.000000 112.000000   0.698848    0.777933        0.529500 0.036833   69.797773 Medium-High     100.000000          False
333-238475 45.000000 118.000000   0.412995    0.678990        0.359100 0.036090   66.893952 Medium-High      99.295775          False
333-191837 44.000000  99.000000   0.999100    0.999100        0.958000 0.036752   66.047227 Medium-High      98.591549          False
 033-87254 47.000000 107.000000   0.550300    0.706433        0.382300 0.036862   64.363680 Medium-High      97.887324          False
 033-25941 48.000000  94.000000   0.998786    0.998786        0.822600 0.036919   64.072742 Medium-High      97.183099          False
 002-94157 51.000000 100.000000   0.490429    0.596624        0.273500 0.037052   64.010906 Medium-High      96.478873          False
 033-48220 57.000000  61.000000   0.999